# Tutorial: Secure Multi-Agent Pipelines Against Runaway Costs and PII Leaks

- **Level**: Intermediate
- **Time to complete**: 20 min
- **Components Used**: [`OpenAIChatGenerator`](https://docs.haystack.deepset.ai/docs/openaichatgenerator), [`Agent`](https://docs.haystack.deepset.ai/docs/agent), [`Pipeline`](https://docs.haystack.deepset.ai/docs/pipeline)
- **Prerequisites**: OpenAI API key
- **Goal**: After completing this tutorial, you will have learned how to add deterministic governance to Haystack pipelines using TealTiger, preventing PII leaks, secret exposure, and runaway LLM costs -- all without adding another LLM to the governance path.

## Overview

Haystack makes it straightforward to build RAG pipelines and multi-agent systems. But once agents can call tools, query databases, or generate code, new risks appear:

- **PII leaks**: An agent retrieves documents containing SSNs or credit card numbers and passes them to the LLM.
- **Secret exposure**: A code-generating agent embeds API keys in its output.
- **Cost runaway**: A research loop keeps calling the LLM indefinitely, burning through budget.

[TealTiger](https://github.com/agentguard-ai/tealtiger) is a deterministic governance SDK that intercepts these problems *before* they reach the LLM or the user. No LLM in the governance path, under 5ms overhead, and structured audit evidence for every decision.

In this tutorial, you will:
1. Build a simple RAG pipeline with Haystack
2. Add TealTiger governance to detect PII and secrets before they reach the model
3. Enforce a per-session cost budget
4. Inspect the governance audit trail

## Installing Dependencies

In [ ]:
%%bash
pip install haystack-ai tealtiger

## Setting Up the Environment

You need an OpenAI API key to run this tutorial. Enter it when prompted below.

In [ ]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## Understanding the Problem: Ungoverned Pipelines

Let's first see what happens in a pipeline *without* governance. Imagine a document store that contains records with personally identifiable information:

In [ ]:
from haystack import Document

# Simulated documents that contain sensitive data
documents = [
    Document(
        content="Customer John Smith, SSN: 123-45-6789, has an outstanding balance of $4,200.",
        meta={"source": "customer_records"},
    ),
    Document(
        content="The database connection uses password: sk-prod-8f2a9b4c1d3e5f6a7b8c9d0e and connects to prod-db.internal.company.com.",
        meta={"source": "internal_docs"},
    ),
    Document(
        content="Q4 revenue reached $12.3M, a 15% increase year-over-year. The engineering team expanded to 45 members.",
        meta={"source": "earnings_report"},
    ),
]

print(f"Loaded {len(documents)} documents")
print(f"Document 1 contains SSN: {'123-45' in documents[0].content}")
print(f"Document 2 contains a secret: {'sk-prod' in documents[1].content}")

In an ungoverned pipeline, all of this content flows directly to the LLM -- the SSN, the API key, everything. The model sees it, it may appear in logs, and there is no audit trail proving governance evaluated the content.

## Adding TealTiger Governance

TealTiger wraps LLM clients (like OpenAI) with a governance layer. It scans inputs and outputs for PII, secrets, and policy violations *before* they reach the model.

In [ ]:
from tealtiger import TealOpenAI, TealOpenAIConfig
from tealtiger.guardrails import PII_PATTERNS, SECRET_PATTERNS

# Create a governed OpenAI client
governed_client = TealOpenAI(
    config=TealOpenAIConfig(
        api_key=os.environ["OPENAI_API_KEY"],
        guardrails={
            "pii_detection": True,       # Block SSN, credit cards, etc.
            "secret_detection": True,    # Block API keys, tokens
            "prompt_injection": True,    # Block injection attempts
        },
        budget={
            "max_cost_per_session": 0.50,  # Hard cap: $0.50 per session
        },
    )
)

print("TealTiger governed client created.")
print(f"  PII detection: enabled ({len(PII_PATTERNS)} patterns)")
print(f"  Secret detection: enabled ({len(SECRET_PATTERNS)}+ patterns)")
print(f"  Budget limit: $0.50/session")

## Building a Governed RAG Pipeline

Now let's build a Haystack RAG pipeline that uses the governed client. TealTiger integrates at the LLM client level, so it works transparently with any Haystack component that calls OpenAI.

In [ ]:
from haystack import Pipeline
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.dataclasses import ChatMessage

# Build the RAG pipeline with TealTiger governance
rag_pipeline = Pipeline()

# Use Haystack's OpenAIChatGenerator -- TealTiger governs at the API level
rag_pipeline.add_component(
    "prompt_builder",
    ChatPromptBuilder(
        template=[
            ChatMessage.from_user(
                "Based on the following context, answer the question.\n"
                "Context: {{context}}\n"
                "Question: {{question}}"
            )
        ]
    ),
)

rag_pipeline.add_component(
    "llm",
    OpenAIChatGenerator(model="gpt-4o-mini"),
)

rag_pipeline.connect("prompt_builder", "llm")

print("RAG pipeline built successfully.")

## Testing: Safe Document (No PII)

Let's first query with the earnings report document -- this contains no sensitive data and should pass governance.

In [ ]:
# Query with clean document -- should pass governance
safe_context = documents[2].content  # Earnings report

result = rag_pipeline.run(
    {
        "prompt_builder": {
            "question": "What was the Q4 revenue?",
            "context": safe_context,
        }
    }
)

print("Query: What was the Q4 revenue?")
print(f"Answer: {result['llm']['replies'][0].text}")
print("\nGovernance: ALLOWED (no PII or secrets detected)")

## Testing: PII Detection in Action

Now let's use TealTiger's standalone scanning to show what happens when PII is present in the pipeline context. In a production setup, you would add a TealTiger pre-processing step before the LLM call.

In [ ]:
from tealtiger import TealGuard

# Create a governance guard for content scanning
guard = TealGuard(
    guardrails=["pii_detection", "secret_detection"]
)

# Scan the document with PII
pii_context = documents[0].content  # Contains SSN
scan_result = guard.scan(pii_context)

print("Scanning document with PII...")
print(f"  Content: {pii_context[:50]}...")
print(f"  Decision: {scan_result.action}")
print(f"  Reason: {scan_result.reason_codes}")
print(f"  Risk Score: {scan_result.risk_score}")
print(f"  Latency: {scan_result.evaluation_time_ms:.2f}ms")
print()

# Scan the document with secrets
secret_context = documents[1].content  # Contains API key
scan_result_2 = guard.scan(secret_context)

print("Scanning document with secrets...")
print(f"  Content: {secret_context[:50]}...")
print(f"  Decision: {scan_result_2.action}")
print(f"  Reason: {scan_result_2.reason_codes}")
print(f"  Risk Score: {scan_result_2.risk_score}")
print(f"  Latency: {scan_result_2.evaluation_time_ms:.2f}ms")

TealTiger detected both the SSN and the API key pattern in under 5ms each -- deterministically, with no LLM call in the governance path.

## Building a Governed Pre-Processing Component

To integrate TealTiger directly into a Haystack pipeline, you can create a custom component that scans content before it reaches the LLM:

In [ ]:
from haystack import component
from typing import List, Optional


@component
class TealTigerGovernanceFilter:
    """Haystack component that filters documents through TealTiger governance.

    Scans documents for PII, secrets, and policy violations before they
    reach the LLM. Blocked documents are removed from the pipeline context.
    """

    def __init__(self, guardrails: Optional[List[str]] = None):
        self.guard = TealGuard(
            guardrails=guardrails or ["pii_detection", "secret_detection"]
        )
        self.blocked_count = 0
        self.passed_count = 0

    @component.output_types(documents=List[Document], blocked=List[Document])
    def run(self, documents: List[Document]):
        passed = []
        blocked = []

        for doc in documents:
            result = self.guard.scan(doc.content)
            if result.action == "allow":
                passed.append(doc)
                self.passed_count += 1
            else:
                blocked.append(doc)
                self.blocked_count += 1

        return {"documents": passed, "blocked": blocked}


# Test the component
governance_filter = TealTigerGovernanceFilter()
filter_result = governance_filter.run(documents=documents)

print(f"Documents passed governance: {len(filter_result['documents'])}")
print(f"Documents blocked: {len(filter_result['blocked'])}")
print()
for doc in filter_result["documents"]:
    print(f"  PASSED: {doc.content[:60]}...")
for doc in filter_result["blocked"]:
    print(f"  BLOCKED: {doc.content[:60]}...")

## Enforcing Cost Budgets

TealTiger tracks LLM costs per session and enforces hard limits. This prevents infinite agent loops from burning through your API budget.

In [ ]:
from tealtiger import CostTracker, CostTrackerConfig, InMemoryCostStorage

# Set up cost tracking with a $0.50 session budget
cost_tracker = CostTracker(
    config=CostTrackerConfig(
        storage=InMemoryCostStorage(),
        budget_per_session=0.50,
        budget_per_request=0.10,
    )
)

# Simulate tracking costs
cost_tracker.record(model="gpt-4o-mini", input_tokens=500, output_tokens=200)
cost_tracker.record(model="gpt-4o-mini", input_tokens=800, output_tokens=300)

print(f"Session cost so far: ${cost_tracker.session_cost:.4f}")
print(f"Budget remaining: ${cost_tracker.budget_remaining:.4f}")
print(f"Budget limit: ${cost_tracker.config.budget_per_session:.2f}")
print(f"Within budget: {cost_tracker.is_within_budget()}")

## Inspecting the Governance Audit Trail

Every TealTiger governance decision produces a structured receipt. These can be exported as SARIF (for security tools), JUnit XML (for CI/CD), or JSON (for dashboards).

In [ ]:
import json

# Get the governance audit trail from our earlier scans
audit_trail = guard.get_decisions()

print(f"Total governance decisions recorded: {len(audit_trail)}")
print()

for i, decision in enumerate(audit_trail, 1):
    print(f"Decision {i}:")
    print(f"  ID: {decision.decision_id}")
    print(f"  Action: {decision.action}")
    print(f"  Reason: {decision.reason_codes}")
    print(f"  Risk Score: {decision.risk_score}")
    print(f"  Latency: {decision.evaluation_time_ms:.2f}ms")
    print(f"  Timestamp: {decision.timestamp}")
    print()

## Before and After Comparison

| Concern | Without TealTiger | With TealTiger |
|---------|------------------|----------------|
| PII in context | Flows directly to LLM | Blocked before model sees it |
| Secrets in output | Returned to user | Detected and redacted |
| Cost control | No limit, infinite loops possible | Hard budget cap per session |
| Audit trail | None | Structured evidence for every decision |
| Governance overhead | N/A | <5ms deterministic, no LLM |
| Compliance evidence | Manual log review | SARIF/JUnit XML/JSON export |

## What's next

- Explore the [TealTiger documentation](https://docs.tealtiger.ai/) for advanced policy configuration.
- Check out [TealTiger's Haystack integration package](https://github.com/agentguard-ai/tealtiger/tree/main/packages/tealtiger-python) for deeper pipeline integration.
- Learn about [TealEngine policy modes](https://docs.tealtiger.ai/engine/) (ENFORCE, MONITOR, REPORT_ONLY) for gradual rollout.
- See [Tutorial 45: Creating a Multi-Agent System](https://haystack.deepset.ai/tutorials/45_creating_a_multi_agent_system) to combine agents with governance.

## About us


🎉 Congratulations! You've completed this tutorial!

To stay up to date on the latest Haystack developments, you can [sign up for our newsletter](https://landing.deepset.ai/haystack-community-updates) or [join Haystack Discord community](https://discord.com/invite/xYvH6drSmA).

Get in touch:
[X](https://x.com/Haystack_AI) | [LinkedIn](https://www.linkedin.com/showcase/haystack-ai-framework/) | [Discord](https://discord.com/invite/xYvH6drSmA) | [GitHub Discussions](https://github.com/deepset-ai/haystack/discussions) | [Haystack Website](https://haystack.deepset.ai)
